# Nasdaq ITCH Feed Handler v3.1

This notebook is the ZCU106 pre-Taxi hardware regression for the **native 64-bit ingress**.
In its default state, it tracks Apple, Microsoft, and Netflix using Nasdaq ITCH historical
data from 30/12/2019.

The hardware path exercised here is still PS DDR -> AXI DMA -> PL, but the AXI DMA MM2S
stream and ingress are now 64-bit. This lets us prove the migrated ingress on the board
before replacing the DMA source with Taxi 10GbE.


In [ ]:
# imports

from pynq import Overlay, allocate, MMIO
import numpy as np
import time
import gzip
import struct
import math
import golden.runner as test
import json
import pandas as pd
import urllib.request
import io
import ssl


### Key Note before use:

For the variables below, you are free to change them, however, there are some notes you
should be aware of if you are trying to change these variables.

- Place `v3_1.bit` and the corresponding `v3_1.hwh` in the same directory. PYNQ will use
  the matching HWH when loading the overlay.
- The `SW_TARGET_SYMBOL` can only be used **once** to track **one stock**. Ensure that you
  pick one stock that you are also tracking in hardware.
- Ensure that the stock you track is actually in Nasdaq for the date of the historical data.
  For the hardware stock ensure that there are **4 spaces after the last letter of the
  symbol**, or the system will fail.
- Base price values are given for the default stocks. Wrapping (values below the base price)
  is **not supported**, so ensure that the base price is **below the minimum price you expect
  for the historical data**.
- Ensure the data path includes `localhost:8443`, or the relevant port used to open the
  reverse SSH tunnel.
- v3.1 uses a **64-bit MM2S/ingress stream**. The DMA backing buffer therefore uses
  `np.uint64`, and packet storage is padded to an 8-byte boundary. For this pre-Taxi board
  regression the padded final 64-bit word is transmitted in full, because the DMA produces
  LSB-contiguous partial-beat `TKEEP` while the ingress uses an MSB-contiguous byte-lane
  convention. The UDP length still defines the true MoldUDP64 payload length, so the extra
  bytes are treated as harmless Ethernet-frame padding.


In [ ]:
PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/v3_1.bit"
DATA_URL    = "https://localhost:8443/ITCH/Nasdaq%20ITCH/12302019.NASDAQ_ITCH50.gz"

SW_MESSAGES_TO_READ = 1_000_000 # number of messages read in software - Recommended to keep below this number - data extraction may take a very long time otherwise

MSG_LIMIT        = True # If true, hardware and software both process SW_MESSAGES_TO_READ source messages


SW_TARGET_SYMBOL = "MSFT" # ensure this matches a Hardware value

HW_SYMBOL_0      = b"AAPL    " # This is 4 spaces after the L - required for accurate data
HW_SYMBOL_1      = b"MSFT    "
HW_SYMBOL_2      = b"NFLX    "

BASE_PRICE_STOCK_0  = 0x5208 # base price of $210.00 (for apple)
BASE_PRICE_STOCK_1  = 0x2EE0 # base price of $120.00 (for microsoft)
BASE_PRICE_STOCK_2  = 0x7D00 # base price of $320.00 (for Netflix)


This function is used to generate network headers to pass to the system. The historical
BinaryFILE data does not contain these headers, but we include them so the network ingress
is tested exactly as before.


In [ ]:
# network header gen function

def generate_network_headers(payload_len, seq_num):

    mold_len = 20 + 2 + payload_len
    udp_len = 8 + mold_len
    ip_total_len = 20 + udp_len

    # Ethernet - 14 bytes of form: Dest MAC, Src MAC, EtherType (0x0800 for IPv4)
    eth = struct.pack('!6s6sH', b'\x00\x11\x22\x33\x44\x55', b'\xAA\xBB\xCC\xDD\xEE\xFF', 0x0800)

    # 2. IPv4 - 20 bytes of form: VHL, TOS, TotalLen, ID, Flags/Frag, TTL, Protocol (17=UDP), Checksum, SrcIP, DestIP
    ip = struct.pack('!BBHHHBBH4s4s',
                     0x45, 0x00, ip_total_len, 0x0000, 0x0000,
                     64, 17, 0x0000,
                     b'\xc0\xa8\x01\x64', b'\xc0\xa8\x01\xc8')

    # 3. UDP - 8 bytes of form: SrcPort, DestPort, Length, Checksum
    udp = struct.pack('!HHHH', 12345, 12345, udp_len, 0x0000)

    # 4. MoldUDP64 - 20 bytes of form: Session (10 bytes), Sequence Number (8 bytes), Message Count (2 bytes)
    mold = struct.pack('!10sQH', b'SESSION123', seq_num, 1)

    # 5. Mold Message Block Length 2 bytes
    msg_len_hdr = struct.pack('!H', payload_len)

    return eth + ip + udp + mold + msg_len_hdr


This function is used to obtain the gzip file from the URL. If successful, there will be a
remote connection via a reverse SSH tunnel. Otherwise, the file can be opened locally.


In [ ]:
def get_gzip_stream(url_or_path):

    if url_or_path.startswith(("http://", "https://")):
        print("Enabling remote connection")
        ssl_ctx = ssl.create_default_context()
        ssl_ctx.check_hostname = False
        ssl_ctx.verify_mode = ssl.CERT_NONE
        req = urllib.request.Request(
            url_or_path,
            headers={
                "Host": "emi.nasdaq.com",
                "User-Agent": "Mozilla/5.0",
            },
        )
        response = urllib.request.urlopen(req, context=ssl_ctx)
        gz_file = gzip.GzipFile(fileobj=response)
        print("Remote connection successful!")
        return io.BufferedReader(gz_file, buffer_size=1024 * 1024), response
    else:
        print("Enabling local download")
        ## Open file locally if remote connection fails
        gz_file = gzip.open(url_or_path, "rb")
        print("Download complete!")
        return io.BufferedReader(gz_file, buffer_size=1024 * 1024), None


Download the overlay for hardware.


In [ ]:
# Overlay Load

ol = Overlay(PATH)
ol.download()
print("Overlay Loaded:")


In [ ]:
gpio_bid   = ol.axi_gpio_bbo_bid
gpio_ask   = ol.axi_gpio_bbo_ask
gpio_base_price1 = ol.axi_gpio_price
gpio_base_price2 = ol.axi_gpio_price1
gpio_stock_id    = ol.axi_gpio_meta
dma = ol.axi_dma_0
print("All IPs Loaded")


Variables and other declarations for hardware use.


In [ ]:
# v3.1: the MM2S AXI stream is 64-bit, so use 64-bit backing words.
send_buffer = allocate(shape=(256,), dtype=np.uint64)

HW_MESSAGES_TO_READ = SW_MESSAGES_TO_READ

files = {
    1: open(f"hw_{HW_SYMBOL_0[0:4].decode()}_data.txt", "w"),
    2: open(f"hw_{HW_SYMBOL_1[0:4].decode()}_data.txt", "w"),
    3: open(f"hw_{HW_SYMBOL_2[0:4].decode()}_data.txt", "w")
}

json_files = {
    1: open(f"hw_{HW_SYMBOL_0[0:4].decode()}_data.jsonl", "w"),
    2: open(f"hw_{HW_SYMBOL_1[0:4].decode()}_data.jsonl", "w"),
    3: open(f"hw_{HW_SYMBOL_2[0:4].decode()}_data.jsonl", "w")
}

target_symbols = {
    HW_SYMBOL_0[0:4]: 1,
    HW_SYMBOL_1[0:4]: 2,
    HW_SYMBOL_2[0:4]: 3,
}

target_locate_0   = None
target_locate_1   = None
target_locate_2   = None

gpio_base_price1.channel1.write(BASE_PRICE_STOCK_0, 0xffff_ffff)
gpio_base_price1.channel2.write(BASE_PRICE_STOCK_1, 0xffff_ffff)
gpio_base_price2.channel1.write(BASE_PRICE_STOCK_2, 0xffff_ffff)

# Trackers
msg_count = 0
target_msg_count = 0
last_bid_price = {1: 0, 2: 0, 3: 0}
last_ask_price = {1: 0, 2: 0, 3: 0}
last_bid_shares = {1: 0, 2: 0, 3: 0}
last_ask_shares = {1: 0, 2: 0, 3: 0}
locate_to_stock_id = {}


if not dma.sendchannel.running:
    dma.sendchannel.start()


## Hardware run

This runs the hardware design and outputs data in the `hw_*_data` files.

The text files are readable output for the user.

The JSONL files are used for the Hardware/Software Comparison.

This is a **correctness regression**, not a throughput benchmark: each DMA transfer is waited
on before the next message. That keeps the board test deterministic while we validate the new
64-bit ingress.


In [ ]:
stream_render, net_response = get_gzip_stream(DATA_URL)

try:
    with stream_render as data:
        while True:
            start_msg = data.read(2) # Reading the 2-byte length header
            if not start_msg: break

            ins_len = int.from_bytes(start_msg, byteorder='big')
            output_data = data.read(ins_len)
            if len(output_data) < ins_len: break

            msg_count += 1
            if (msg_count > HW_MESSAGES_TO_READ) and MSG_LIMIT:
                print("Message Limit Reached")
                if net_response:
                    net_response.close()
                    print("Remote connection closed")
                for f in files.values():
                    f.close()
                break
            msg_type = output_data[0:1]
            locate_code = output_data[1:3]

            if msg_type == b'R':
                symbol = output_data[11:15]

                if symbol in target_symbols:
                    s_id = target_symbols[symbol]
                    locate_to_stock_id[locate_code] = s_id
                    print(f"Stock {symbol.decode()} mapped to ID {s_id} (Locate: {locate_code.hex()})")

            # Message modify step for Hardware
            modified_msg = bytearray(output_data)
            keep_message = False

            s_id = locate_to_stock_id.get(locate_code)

            if msg_type == b'S': keep_message = True
            elif s_id is not None:
                modified_msg[1:3] = s_id.to_bytes(2, byteorder='big')

                # Price Division for hardware use
                offset = 32 if msg_type in [b'A', b'F', b'C'] else (31 if msg_type == b'U' else None)
                if offset:
                    price = int.from_bytes(modified_msg[offset:offset+4], 'big') // 100
                    modified_msg[offset:offset+4] = price.to_bytes(4, 'big')
                keep_message = True


            if keep_message:

                # Generate the 64-byte network header chain
                network_headers = generate_network_headers(payload_len=ins_len, seq_num=msg_count)

                full_packet = network_headers + modified_msg

                # v3.1: use 8-byte backing-word alignment for the 64-bit DMA.
                # Each 64-bit word is byte-swapped below to match the ingress convention where
                # the first network byte occupies TDATA[63:56]. A partial AXI DMA transfer would
                # generate an LSB-contiguous TKEEP, whereas the ingress expects MSB-contiguous
                # valid lanes. Therefore transmit the final padded 64-bit word in full.
                #
                # The extra zero bytes are harmless Ethernet-frame padding: frame_crack uses the
                # UDP length to delimit the MoldUDP64 datagram and drains any bytes after it.
                packet_len = len(full_packet)
                storage_len = math.ceil(packet_len / 8) * 8
                padded = full_packet.ljust(storage_len, b'\x00')

                # Big Endian swap across each native 64-bit stream word.
                temp_arr = np.frombuffer(padded, dtype=np.uint64)
                send_buffer.fill(0)
                send_buffer[:len(temp_arr)] = temp_arr.byteswap()

                dma.sendchannel.transfer(send_buffer[:len(temp_arr)], nbytes=storage_len)
                dma.sendchannel.wait()

                if msg_type != b'S' and s_id in last_bid_price:

                    hw_s_id = gpio_stock_id.channel1.read()

                    if hw_s_id in last_bid_price:
                        bid_price = gpio_bid.channel1.read()
                        ask_price = gpio_ask.channel1.read()
                        bid_shares = gpio_bid.channel2.read()
                        ask_shares = gpio_ask.channel2.read()

                        # Capture updates even if only the shares change
                        if (bid_price != last_bid_price[hw_s_id] or
                            ask_price != last_ask_price[hw_s_id] or
                            bid_shares != last_bid_shares[hw_s_id] or
                            ask_shares != last_ask_shares[hw_s_id]):

                            last_bid_price[hw_s_id], last_ask_price[hw_s_id] = bid_price, ask_price
                            last_bid_shares[hw_s_id], last_ask_shares[hw_s_id] = bid_shares, ask_shares

                            bid_formatted = f"${(last_bid_price[hw_s_id] / 100.0):.2f}"
                            ask_formatted = f"${(last_ask_price[hw_s_id] / 100.0):.2f}"

                            files[hw_s_id].write(
                                f"{last_bid_shares[hw_s_id]:>4} Bid shares at price: {bid_formatted:>8} | "
                                f"{last_ask_shares[hw_s_id]:>4} Ask shares at price: {ask_formatted:>8}\n"
                            )


                            bbo_data = {"bbo": {"ask_price": last_ask_price[hw_s_id]*100, "ask_size": last_ask_shares[hw_s_id],
                                                "bid_price": last_bid_price[hw_s_id]*100, "bid_size": last_bid_shares[hw_s_id]}}

                            json_data = json.dumps(bbo_data)
                            json_files[hw_s_id].write(json_data + "\n")

finally:
    print("All data obtained")
    if net_response:
        net_response.close()
        print("Remote connection closed")
    for f in files.values():
        f.close()
    for f in json_files.values():
        f.close()


## Software run

This run uses the golden model with the same historical data. By default we only read
1,000,000 messages (this can be altered). Relevant data in JSON form is written to
`golden_states_<symbol>.jsonl`.

The golden model is byte/message based and is intentionally unchanged for v3.1; the
32-bit -> 64-bit migration is a hardware transport change, not an ITCH semantic change.


In [ ]:
# Test with golden model

subsection = bytearray()

stream_render, net_response = get_gzip_stream(DATA_URL)

with stream_render as data:
    for msg in range(SW_MESSAGES_TO_READ):
        start_msg = data.read(2) # Reading the 2-byte length header
        if not start_msg:
            break

        ins_len = int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)
        if len(output_data) < ins_len:
            break

        subsection.extend(start_msg)
        subsection.extend(output_data)

if net_response:
        net_response.close()
        print("Remote connection closed")
print(f"Extracted {SW_MESSAGES_TO_READ} messages. Running golden model...")

with open(f"golden_events_{SW_TARGET_SYMBOL}.jsonl", "w", encoding="utf-8") as events_out, \
     open(f"golden_states_{SW_TARGET_SYMBOL}.jsonl", "w", encoding="utf-8") as states_out:

    # run_bytes natively accepts bytearray objects
    test.run_bytes(
        data=subsection,
        events_out=events_out,
        states_out=states_out,
        symbol=SW_TARGET_SYMBOL
    )

print("Subsection processing complete!")


## Hardware/Software Comparison

This cell compares the software and hardware **BBO-change sequences** over the same source-message range.

The comparison accounts for two intentional hardware representation constraints:

- the golden model represents an empty book side as `None`, while the hardware GPIOs expose it as `0`;
- BBO states below the configured hardware base price are not representable by the current order-book implementation. These are reported and skipped explicitly.

Any genuine semantic mismatches are written to `error_log.txt`. Skipped out-of-range golden states are written to `comparison_skipped.jsonl`.


In [ ]:
# Data comparisons

HW_TARGET_FILE = f"hw_{SW_TARGET_SYMBOL}_data.jsonl"
SW_TARGET_FILE = f"golden_states_{SW_TARGET_SYMBOL}.jsonl"

BASE_PRICE_BY_SYMBOL = {
    "AAPL": BASE_PRICE_STOCK_0,
    "MSFT": BASE_PRICE_STOCK_1,
    "NFLX": BASE_PRICE_STOCK_2,
}

base_price_cents = BASE_PRICE_BY_SYMBOL[SW_TARGET_SYMBOL]


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                records.append(json.loads(line))
    return records


def normalise_bbo(bbo):
    """Return (bid_price, bid_size, ask_price, ask_size), using zero for an empty side."""
    return (
        0 if bbo["bid_price"] is None else int(bbo["bid_price"]),
        0 if bbo["bid_size"] is None else int(bbo["bid_size"]),
        0 if bbo["ask_price"] is None else int(bbo["ask_price"]),
        0 if bbo["ask_size"] is None else int(bbo["ask_size"]),
    )


def bbo_is_representable(bbo):
    """Check whether each non-empty BBO price is at/above the configured HW base price."""
    bid_price, bid_size, ask_price, ask_size = normalise_bbo(bbo)

    if bid_size != 0 and (bid_price // 100) < base_price_cents:
        return False

    if ask_size != 0 and (ask_price // 100) < base_price_cents:
        return False

    return True


software_records = load_jsonl(SW_TARGET_FILE)
hardware_records = load_jsonl(HW_TARGET_FILE)

# The golden runner writes a state after every relevant event.
# The notebook hardware logger writes only when the observed BBO changes,
# so reduce the golden output to the same BBO-change contract first.
golden_changes = []
last_bbo = None

for record in software_records:
    current_bbo = normalise_bbo(record["bbo"])

    if current_bbo != last_bbo:
        golden_changes.append({
            "msg_index": int(record["msg_index"]),
            "raw_bbo": record["bbo"],
            "bbo": current_bbo,
        })
        last_bbo = current_bbo


# Explicitly report golden states which the current hardware price window cannot represent.
comparable_golden = []
skipped_golden = []

for entry in golden_changes:
    if bbo_is_representable(entry["raw_bbo"]):
        comparable_golden.append(entry)
    else:
        skipped_golden.append(entry)


hardware_changes = [
    {
        "hw_index": index,
        "bbo": normalise_bbo(record["bbo"]),
    }
    for index, record in enumerate(hardware_records)
]


with open("comparison_skipped.jsonl", "w", encoding="utf-8") as skipped_out:
    for entry in skipped_golden:
        skipped_out.write(json.dumps({
            "msg_index": entry["msg_index"],
            "bbo": {
                "bid_price": entry["bbo"][0],
                "bid_size": entry["bbo"][1],
                "ask_price": entry["bbo"][2],
                "ask_size": entry["bbo"][3],
            },
            "reason": (
                f"BBO contains a non-empty price below the configured "
                f"{SW_TARGET_SYMBOL} hardware base of {base_price_cents} cents"
            ),
        }) + "\n")


compare_count = min(len(comparable_golden), len(hardware_changes))
mismatches = []

for index in range(compare_count):
    sw_entry = comparable_golden[index]
    hw_entry = hardware_changes[index]

    if sw_entry["bbo"] != hw_entry["bbo"]:
        mismatches.append({
            "compare_index": index,
            "golden_msg_index": sw_entry["msg_index"],
            "golden_bbo": sw_entry["bbo"],
            "hardware_index": hw_entry["hw_index"],
            "hardware_bbo": hw_entry["bbo"],
        })


missing_hardware = max(0, len(comparable_golden) - len(hardware_changes))
extra_hardware = max(0, len(hardware_changes) - len(comparable_golden))

with open("error_log.txt", "w", encoding="utf-8") as error:
    for mismatch in mismatches:
        error.write(
            f"Comparison index {mismatch['compare_index']}: "
            f"golden msg_index={mismatch['golden_msg_index']} "
            f"golden={mismatch['golden_bbo']} "
            f"hardware index={mismatch['hardware_index']} "
            f"hardware={mismatch['hardware_bbo']}\n"
        )

    if missing_hardware:
        error.write(
            f"Hardware sequence ended early: missing {missing_hardware} "
            "comparable BBO change(s).\n"
        )

    if extra_hardware:
        error.write(
            f"Hardware sequence has {extra_hardware} unexpected extra "
            "BBO change(s).\n"
        )


semantic_errors = len(mismatches) + missing_hardware + extra_hardware

print("Comparison Complete!\n")
print(f"Golden state rows:             {len(software_records)}")
print(f"Golden BBO changes:            {len(golden_changes)}")
print(f"Skipped out-of-range changes:  {len(skipped_golden)}")
print(f"Comparable golden BBO changes: {len(comparable_golden)}")
print(f"Hardware BBO changes:          {len(hardware_changes)}")
print(f"Semantic mismatches:           {len(mismatches)}")
print(f"Missing hardware changes:      {missing_hardware}")
print(f"Unexpected HW changes:         {extra_hardware}")
print(f"Total comparison errors:       {semantic_errors}")

if skipped_golden:
    print()
    print(
        f"Note: {len(skipped_golden)} golden BBO change(s) were outside the "
        f"configured {SW_TARGET_SYMBOL} hardware price range and are listed in "
        "`comparison_skipped.jsonl`."
    )

if semantic_errors == 0:
    print("\nPASS: hardware BBO sequence matches the comparable golden sequence.")
else:
    print("\nFAIL: see error_log.txt for the first level of detail.")


### Optional mismatch detail

Run this only if the comparison above reports a non-zero error count. It prints the first mismatches without rerunning either hardware or the golden model.


In [ ]:
# Optional detailed mismatch output

if semantic_errors == 0:
    print("No semantic mismatches to inspect.")
else:
    print("First semantic mismatches:\n")

    for mismatch in mismatches[:20]:
        print(
            f"idx={mismatch['compare_index']} | "
            f"golden_msg={mismatch['golden_msg_index']} | "
            f"SW={mismatch['golden_bbo']} | "
            f"HW={mismatch['hardware_bbo']}"
        )

    if missing_hardware:
        print(f"\nHardware is missing {missing_hardware} BBO change(s) at the end.")

    if extra_hardware:
        print(f"\nHardware has {extra_hardware} unexpected extra BBO change(s) at the end.")
